# 🎬 OpenSource Clipping — Colab (T4 + Python 3.12 + pip)

Notebook para rodar o projeto **do zero** no Google Colab, usando:

- **GPU T4** (`Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU`)
- **venv com Python 3.12** criada com `python3.12 -m venv` e dependências instaladas com **pip** (sem uv)

Ele já resolve os problemas encontrados até aqui:

| Problema | Onde é resolvido |
|---|---|
| `ffmpeg is not installed` | Passo 2 |
| `No supported JavaScript runtime` (yt-dlp) | Passo 2 (Deno) + Passo 4 (`yt-dlp[default]`) |
| `libcublas.so.12 is not found` | Passo 4 (libs CUDA 12 na venv) + Passo 5 (`LD_LIBRARY_PATH`) |
| `Sign in to confirm you're not a bot` | Passo 8 (cookies) ou fonte `gdrive` |

> ▶️ Rode as células **em ordem**, de cima para baixo. Se o ambiente de execução for
> reiniciado/desconectado, rode tudo de novo a partir do Passo 1 (os passos de instalação
> são rápidos na segunda vez se a VM não tiver sido recriada).

## 1. Verificar GPU e clonar o repositório

In [ ]:
# Confirma que a T4 está ativa. Se der erro, troque o ambiente de execução para T4 GPU.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import os

REPO_URL = "https://github.com/anderson-nascimento/opensource-clipping.git"
REPO_DIR = "/content/opensource-clipping"
VENV_DIR = "/content/venv312"
VENV_PY  = f"{VENV_DIR}/bin/python"

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("📁 Repositório já existe — atualizando...")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("📂 Diretório atual:", os.getcwd())

## 2. Dependências do sistema: FFmpeg e Deno

- **FFmpeg**: usado pelo yt-dlp (juntar vídeo + áudio) e por toda a renderização.
- **Deno**: runtime JavaScript que o yt-dlp usa para resolver os desafios do YouTube.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -n 1

!curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null
if "/root/.deno/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = "/root/.deno/bin:" + os.environ["PATH"]
!deno --version | head -n 1

## 3. Criar a venv com Python 3.12

O Python padrão do Colab pode não ser 3.12, então instalamos o 3.12 pelo apt
(usando o PPA *deadsnakes* se necessário) e criamos uma venv isolada em `/content/venv312`.

In [ ]:
%%bash
set -e
if ! command -v python3.12 > /dev/null || ! python3.12 -c "import ensurepip" 2> /dev/null; then
  apt-get -qq update
  if ! apt-get -qq install -y python3.12 python3.12-venv python3.12-dev > /dev/null 2>&1; then
    echo "python3.12 não está no apt padrão — adicionando PPA deadsnakes..."
    apt-get -qq install -y software-properties-common > /dev/null
    add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
    apt-get -qq update
    apt-get -qq install -y python3.12 python3.12-venv python3.12-dev > /dev/null
  fi
fi
python3.12 --version

In [ ]:
if not os.path.exists(VENV_PY):
    !python3.12 -m venv {VENV_DIR}

!{VENV_PY} --version
!{VENV_PY} -m pip install -q --upgrade pip setuptools wheel
!{VENV_PY} -m pip --version

## 4. Instalar dependências Python (pip, dentro da venv)

Leva alguns minutos (torch, ultralytics, pyannote...). Além do `requirements.txt`, instala:

- `yt-dlp[default]` → inclui o componente EJS para os desafios JS do YouTube
- `nvidia-cublas-cu12` e `nvidia-cudnn-cu12==9.*` → bibliotecas CUDA 12 que o **faster-whisper/ctranslate2** precisa (`libcublas.so.12`)

In [ ]:
!{VENV_PY} -m pip install -q -r requirements.txt
!{VENV_PY} -m pip install -q -U "yt-dlp[default]"
!{VENV_PY} -m pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!{VENV_PY} -m pip install -q "av<19"  # PyAV 19 quebra o faster-whisper (metadata_errors)
print("✅ Dependências instaladas")

## 5. Configurar o `LD_LIBRARY_PATH` (bibliotecas CUDA da venv)

Usamos `os.environ` (e não `!export`) porque cada `!` no Colab roda num shell novo —
o `os.environ` é herdado por todos os comandos `!` seguintes.

> ⚠️ Se o ambiente de execução reiniciar, rode esta célula novamente.

In [ ]:
import glob

SITE = glob.glob(f"{VENV_DIR}/lib/python3.12/site-packages")[0]
lib_dirs = sorted(glob.glob(f"{SITE}/nvidia/*/lib"))

for nome in ["libcublas.so.12", "libcudnn.so.9"]:
    ok = glob.glob(f"{SITE}/nvidia/*/lib/{nome}*")
    print(("✅ OK    " if ok else "❌ FALTA "), nome)

atual = os.environ.get("LD_LIBRARY_PATH", "")
novos = [d for d in lib_dirs if d not in atual.split(":")]
os.environ["LD_LIBRARY_PATH"] = ":".join(novos + ([atual] if atual else []))
print(f"\n{len(lib_dirs)} pastas nvidia/*/lib adicionadas ao LD_LIBRARY_PATH")

## 6. Teste rápido: Whisper rodando na GPU

In [ ]:
%%writefile /content/check_gpu.py
import numpy as np
import ctranslate2, torch
from faster_whisper import WhisperModel

print("ctranslate2", ctranslate2.__version__, "| GPUs CUDA:", ctranslate2.get_cuda_device_count())
print("torch", torch.__version__, "| cuda disponível:", torch.cuda.is_available())

# Carrega um modelo pequeno e transcreve 1s de silêncio — isso força o uso do cuBLAS/cuDNN.
m = WhisperModel("tiny", device="cuda", compute_type="float16")
segs, _ = m.transcribe(np.zeros(16000, dtype=np.float32))
list(segs)
print("✅ faster-whisper funcionando na GPU")

In [ ]:
!MPLBACKEND=Agg {VENV_PY} /content/check_gpu.py

## 7. Configuração

Edite os valores abaixo. Para a **fonte do vídeo**:

- `"youtube"` → link do YouTube (no Colab normalmente precisa de cookies — Passo 8)
- `"gdrive"` → link de um vídeo no Google Drive compartilhado como *"Qualquer pessoa com o link"*.
  **Alternativa mais confiável** se o YouTube bloquear o Colab: baixe o vídeo no seu PC e suba no Drive.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURAÇÃO — edite aqui
# ══════════════════════════════════════════════════════════════════════
VIDEO_URL = "https://www.youtube.com/watch?v=SEU_VIDEO_ID"   # <-- ALTERE
SOURCE    = "youtube"        # "youtube", "gdrive", "tiktok" ou "instagram"

TOTAL_CLIPS = 3

# Whisper — na T4: large-v3 + float16 (se faltar memória: int8_float16 ou modelo "medium")
WHISPER_MODEL        = "large-v3"
WHISPER_DEVICE       = "cuda"
WHISPER_COMPUTE_TYPE = "float16"

# Opcionais
ASPECT_RATIO       = "9:16"     # "9:16", "16:9", "1:1"
FONT_STYLE         = "HORMOZI"  # "DEFAULT", "HORMOZI", "CINEMATIC", "STORYTELLER"
HOOK_DURATION      = 3
WORDS_PER_SUBTITLE = 5

print("✅ Configuração carregada:", SOURCE, VIDEO_URL)

### Chaves de API (Colab Secrets)

1. Clique no ícone 🔑 na barra lateral do Colab
2. Adicione `GOOGLE_API_KEY` (obrigatória — [obter aqui](https://aistudio.google.com/apikey))
3. Opcionais: `PEXELS_API_KEY` (B-roll) e `HF_TOKEN` (split-screen / camera-switch)
4. Ative **"Acesso ao notebook"** para cada uma

In [ ]:
from google.colab import userdata
from pathlib import Path

def _secret(nome):
    try:
        return userdata.get(nome) or ""
    except Exception:
        return ""

GOOGLE_API_KEY = _secret("GOOGLE_API_KEY")
PEXELS_API_KEY = _secret("PEXELS_API_KEY")
HF_TOKEN       = _secret("HF_TOKEN")

Path(f"{REPO_DIR}/.env").write_text(
    f"GOOGLE_API_KEY={GOOGLE_API_KEY}\n"
    f"PEXELS_API_KEY={PEXELS_API_KEY}\n"
    f"HF_TOKEN={HF_TOKEN}\n",
    encoding="utf-8",
)

print("GOOGLE_API_KEY :", "✅" if GOOGLE_API_KEY else "❌ FALTANDO (obrigatória)")
print("PEXELS_API_KEY :", "✅" if PEXELS_API_KEY else "⚪ não definida (opcional)")
print("HF_TOKEN       :", "✅" if HF_TOKEN else "⚪ não definida (opcional)")

## 8. Cookies do YouTube (necessário para YouTube no Colab)

O YouTube bloqueia IPs de datacenter com *"Sign in to confirm you're not a bot"*.
Para passar, envie um `youtube_cookies.txt` **válido**:

1. No seu PC, abra uma **janela anônima** e faça login no YouTube (de preferência uma conta secundária).
2. Na mesma aba, acesse `https://www.youtube.com/robots.txt`.
3. Exporte os cookies com a extensão **"Get cookies.txt LOCALLY"** (formato Netscape) — habilite a extensão no modo anônimo.
4. **Feche a janela anônima** imediatamente (se continuar navegando, o YouTube invalida os cookies).
5. Rode a célula abaixo e envie o arquivo (ou arraste-o para `/content/youtube_cookies.txt`).

Se a fonte for `gdrive`, pule este passo.

In [ ]:
COOKIES_PATH = "/content/youtube_cookies.txt"

if SOURCE == "youtube" and not os.path.isfile(COOKIES_PATH):
    from google.colab import files
    print("📤 Selecione o arquivo de cookies (cookies.txt)...")
    enviados = files.upload()
    if enviados:
        nome = next(iter(enviados))
        os.replace(nome, COOKIES_PATH)

YT_COOKIES_ARGS = []
if os.path.isfile(COOKIES_PATH):
    primeira = open(COOKIES_PATH, encoding="utf-8", errors="ignore").readline()
    if "Netscape HTTP Cookie File" in primeira or "HTTP Cookie File" in primeira:
        print("✅ Cookies válidos (formato Netscape):", COOKIES_PATH)
    else:
        print("⚠️ O arquivo não parece estar no formato Netscape — o yt-dlp pode rejeitá-lo.")
    YT_COOKIES_ARGS = ["--yt-cookies", COOKIES_PATH]
else:
    print("ℹ️ Sem cookies — o download será tentado sem autenticação.")

In [ ]:
# Teste do download (só lista os formatos, não baixa nada).
# Se aparecer a tabela de formatos, o YouTube está liberado. Se der "Sign in to confirm",
# refaça os cookies (Passo 8) ou use SOURCE = "gdrive".
if SOURCE == "youtube":
    _ck = f"--cookies {COOKIES_PATH}" if YT_COOKIES_ARGS else ""
    !{VENV_PY} -m yt_dlp {_ck} --remote-components ejs:github -F "{VIDEO_URL}" | tail -n 25
else:
    print("Fonte não é YouTube — teste pulado.")

## 9. Rodar o pipeline 🎬

Etapas: download → transcrição (Whisper na T4) → análise com Gemini → renderização dos clipes.

In [ ]:
import shlex

os.chdir(REPO_DIR)
args = [
    "--url", VIDEO_URL,
    "--source", SOURCE,
    "--clips", str(TOTAL_CLIPS),
    "--ratio", ASPECT_RATIO,
    "--font-style", FONT_STYLE,
    "--hook-duration", str(HOOK_DURATION),
    "--words-per-sub", str(WORDS_PER_SUBTITLE),
    "--whisper-model", WHISPER_MODEL,
    "--whisper-device", WHISPER_DEVICE,
    "--whisper-compute-type", WHISPER_COMPUTE_TYPE,
    *YT_COOKIES_ARGS,
]
CMD_ARGS = shlex.join(args)
print("▶️", VENV_PY, "main.py", CMD_ARGS, "\n")

# MPLBACKEND=Agg: o Colab define um backend do Jupyter que não existe na venv
!MPLBACKEND=Agg {VENV_PY} main.py {CMD_ARGS}

## 10. Resultados

In [ ]:
OUT_DIR = f"{REPO_DIR}/outputs"
if os.path.isdir(OUT_DIR):
    for raiz, _, arquivos in os.walk(OUT_DIR):
        for f in sorted(arquivos):
            p = os.path.join(raiz, f)
            print(f"{os.path.getsize(p) / 1024 / 1024:8.1f} MB  {os.path.relpath(p, OUT_DIR)}")
else:
    print("❌ Pasta outputs/ não encontrada — verifique os erros do Passo 9.")

In [ ]:
import shutil
from google.colab import files

if os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR):
    shutil.make_archive("/content/clipping_outputs", "zip", OUT_DIR)
    files.download("/content/clipping_outputs.zip")
else:
    print("❌ Nada para baixar.")

---
## 🛠️ Solução de problemas

| Erro | Solução |
|---|---|
| `libcublas.so.12 is not found` | Rode de novo o Passo 5 (após reinício do runtime o `LD_LIBRARY_PATH` se perde) |
| `Sign in to confirm you're not a bot` | Gere cookies novos (Passo 8) ou use `SOURCE = "gdrive"` |
| `No supported JavaScript runtime` | Rode de novo o Passo 2 (Deno no `PATH`) |
| `unexpected keyword argument 'metadata_errors'` | `pip install "av<19"` na venv (já incluso no Passo 4) |
| `CUDA out of memory` | `WHISPER_COMPUTE_TYPE = "int8_float16"` ou `WHISPER_MODEL = "medium"` |
| `MPLBACKEND` / `matplotlib_inline` | Rode o pipeline com `MPLBACKEND=Agg` (já incluso no Passo 9) |
| `GOOGLE_API_KEY` ausente | Adicione em Colab Secrets (🔑) e ative o acesso ao notebook |
| Teste do Passo 6 falha | Confira se o Passo 5 mostrou ✅ para `libcublas` e `libcudnn` |